# Gestion de comptes bancaires

Ce TP modélise des comptes bancaires en objets. Une version procédurale, faite de dictionnaires et de fonctions, vous est donnée : lisez-la et exécutez-la, puis réécrivez-la en objets, exercice après exercice. Chaque exercice s'appuie sur le précédent : si vous bloquez, exécutez sa solution avant de passer au suivant. Le premier exercice, le diagramme de classes, et la dernière étape, l'historique des opérations, sont facultatifs : vous pouvez les sauter. Les montants sont des nombres entiers d'euros.

## Spécifications

- Il existe quatre types de comptes :
  - le compte courant, qui autorise un découvert de 500 € au plus ;
  - le compte joint, un compte courant à deux titulaires ;
  - le compte d'épargne, dont le solde ne peut jamais être négatif ;
  - le livret A, un compte d'épargne dont le solde ne peut pas dépasser son plafond de 22 950 €.
- Tout le monde peut déposer de l'argent sur un compte ; seul l'un de ses titulaires peut en retirer, ou en virer vers un autre compte.
- Un virement n'a lieu que si les deux comptes l'acceptent ; sinon, aucun des deux soldes ne change.
- Un client a au plus un compte de chaque type. Un compte joint compte pour ses deux titulaires.
- Une opération refusée lève une exception qui dit pourquoi.

## La version procédurale

Le code suivant respecte ces règles avec des dictionnaires et des fonctions, sauf la dernière règle sur les clients, qu'il ignore. Exécutez-le, puis la cellule suivante, qui joue un scénario.

In [ ]:
from typing import Any

DECOUVERT_AUTORISE = 500
PLAFOND_LIVRET_A = 22_950


def cree_compte(type_compte: str, *titulaires: str) -> dict[str, Any]:
  return {"type": type_compte, "titulaires": titulaires, "solde": 0}


def verifie_solde(compte: dict[str, Any], nouveau_solde: int) -> None:
  if compte["type"] in ("courant", "joint") and nouveau_solde < -DECOUVERT_AUTORISE:
    raise ValueError(f"Découvert dépassé : le solde passerait à {nouveau_solde} €")
  if compte["type"] in ("épargne", "livret A") and nouveau_solde < 0:
    raise ValueError(f"Solde insuffisant : le solde passerait à {nouveau_solde} €")
  if compte["type"] == "livret A" and nouveau_solde > PLAFOND_LIVRET_A:
    raise ValueError(f"Plafond dépassé : le solde passerait à {nouveau_solde} €")


def verifie_titulaire(compte: dict[str, Any], demandeur: str) -> None:
  if demandeur not in compte["titulaires"]:
    raise ValueError(f"{demandeur} n'est pas titulaire de ce compte")


def deposer(compte: dict[str, Any], montant: int) -> None:
  verifie_solde(compte, compte["solde"] + montant)
  compte["solde"] += montant


def retirer(compte: dict[str, Any], demandeur: str, montant: int) -> None:
  verifie_titulaire(compte, demandeur)
  verifie_solde(compte, compte["solde"] - montant)
  compte["solde"] -= montant


def virer(
  source: dict[str, Any], demandeur: str, cible: dict[str, Any], montant: int
) -> None:
  verifie_titulaire(source, demandeur)
  verifie_solde(source, source["solde"] - montant)
  verifie_solde(cible, cible["solde"] + montant)
  source["solde"] -= montant
  cible["solde"] += montant

In [ ]:
courant = cree_compte("courant", "Isabelle")
epargne = cree_compte("épargne", "Jean")
livret = cree_compte("livret A", "Jean")
joint = cree_compte("joint", "Isabelle", "Jean")
deposer(courant, 1300)
deposer(epargne, 1000)
deposer(livret, 1200)
deposer(joint, 6500)
virer(courant, "Isabelle", joint, 500)
retirer(livret, "Jean", 60)
print(courant["solde"], epargne["solde"], livret["solde"], joint["solde"])

try:
  retirer(epargne, "Jean", 2000)
except ValueError as e:
  print(f"Refusé : {e}")

epargne["solde"] = -1_000_000  # Rien ne l'empêche
print(epargne)

Ce code fonctionne, mais :

- chaque nouveau type de compte ajoute des `if` à `verifie_solde`, et un type mal orthographié (`"Livret A"`) n'y est soumis à aucune règle ;
- rien n'empêche de modifier un solde directement, comme le fait la dernière ligne ;
- aucun montant n'est vérifié : un dépôt de -100 € est un retrait que n'importe qui peut faire, et un retrait de -100 € crédite le compte ;
- toutes les erreurs sont des `ValueError` : l'appelant ne distingue pas un refus prévu par les règles d'une erreur de programmation, ni un découvert dépassé d'un plafond atteint ;
- rien ne relie un client à ses comptes, et la règle d'un compte de chaque type n'est pas appliquée.

Les exercices suivants réécrivent ce code en objets, qui règlent chacun de ces points.

## Le diagramme de classes (facultatif)

Avant d'écrire le code, dessinez le diagramme de classes des spécifications : les types de comptes et leurs relations d'héritage, le client et ses comptes, avec les attributs et les méthodes de chaque classe. Reprenez-le à la fin du TP, pour le comparer au code que vous aurez écrit.

Écrivez-le en Mermaid dans la cellule de texte qui suit (double-cliquez dessus pour la modifier) : un bloc de code ````mermaid` qui commence par la ligne `classDiagram`. JupyterLab (4.1 et suivants) et GitHub affichent ce bloc comme un diagramme ; si votre environnement ne l'affiche pas, le texte du diagramme reste lisible tel quel. Un papier et un crayon conviennent aussi.

**Documentation utile**

- https://mermaid.js.org/syntax/classDiagram.html

*Votre diagramme ici (double-cliquez pour modifier cette cellule)*

### Conseils supplémentaires

Une classe s'écrit `class Nom { ... }`, avec un membre par ligne, précédé de sa visibilité (le cours, « Diagramme de classes : notation ») : `+` public, `#` protégé (`_solde`). Une méthode a des parenthèses, suivies de son type de retour ; un `*` après elle la marque abstraite, un `$` marque un membre de classe, et `<<abstract>>`, en première ligne, une classe abstraite. Les types génériques s'écrivent avec des tildes : `tuple~str~`. Une relation relie deux classes : `Mere <|-- Fille` pour l'héritage, `Tout o-- Partie` pour une agrégation, `Tout *-- Partie` pour une composition, avec les multiplicités entre guillemets et le nom de la relation après `:`. Une ligne qui commence par `%%` est un commentaire. Un compte joint a deux titulaires, et un client au plus un compte de chaque type : d'où les multiplicités ci-dessous.

```mermaid
classDiagram
  class Compte {
    <<abstract>>
    +tuple~str~ titulaires
    #int _solde
    +solde() int
    #_verifie_solde(nouveau_solde)*
    +deposer(montant)
    %% les autres méthodes
  }
  class CompteCourant {
    +int decouvert_autorise$
    #_verifie_solde(nouveau_solde)
  }
  class Client {
    +str nom
    %% ses attributs et ses méthodes
  }
  Compte <|-- CompteCourant
  Client "1..2" o-- "0..4" Compte : comptes
  %% les autres comptes, et leurs relations
```

### Solution

Le diagramme des classes que construit le TP, la banque et l'historique des dernières étapes compris. Les exceptions, une hiérarchie à part, n'y figurent pas.

```mermaid
classDiagram
  class Compte {
    <<abstract>>
    +tuple~str~ titulaires
    #int _solde
    +list~Operation~ historique
    +solde() int
    #_verifie_solde(nouveau_solde)*
    #_verifie_montant(montant)$
    #_verifie_titulaire(demandeur)
    +deposer(montant)
    +retirer(demandeur, montant)
    +virer(demandeur, cible, montant)
  }
  class CompteCourant {
    +str libelle$
    +int decouvert_autorise$
    #_verifie_solde(nouveau_solde)
  }
  class CompteJoint {
    +str libelle$
  }
  class CompteEpargne {
    +str libelle$
    #_verifie_solde(nouveau_solde)
  }
  class LivretA {
    +str libelle$
    +int plafond$
    #_verifie_solde(nouveau_solde)
  }
  class Client {
    +str nom
    +dict comptes
    +ajouter(compte)
    +patrimoine() int
  }
  class Banque {
    +dict TYPES$
    #list~Compte~ _comptes
    +ouvrir(type_compte, clients) Compte
    +#95;#95;len#95;#95;() int
    +#95;#95;iter#95;#95;() Iterator
  }
  class Operation {
    <<dataclass>>
    +str nature
    +int montant
    +int solde
  }
  Compte <|-- CompteCourant
  CompteCourant <|-- CompteJoint
  Compte <|-- CompteEpargne
  CompteEpargne <|-- LivretA
  Client "1..2" o-- "0..4" Compte : comptes
  Banque "1" *-- "*" Compte : _comptes
  Compte "1" *-- "*" Operation : historique
```

Un compte existe sans son client, et un compte joint appartient à deux clients : le client agrège ses comptes. La banque, elle, crée les comptes et les garde (composition), comme chaque compte son historique. Dans `#95;#95;len#95;#95;`, `#95;` est le code du caractère `_` : sans lui, Mermaid afficherait `__len__` en gras.

## Les exceptions métier

Une opération refusée lève une exception métier, d'une classe qui hérite de `MetierError`. La cellule suivante définit cette hiérarchie : exécutez-la. Le suffixe `Error` de leurs noms suit l'usage de la bibliothèque standard (`ValueError`, `KeyError`).

Il y manque `SoldeInsuffisantError`, levée quand le solde deviendrait négatif ou dépasserait le découvert autorisé : écrivez-la. Contrairement aux autres, elle porte des données. On la crée avec le solde du compte et le montant demandé, `SoldeInsuffisantError(solde, montant)` ; elle les range dans ses attributs `solde` et `montant`, et construit son message à partir d'eux, par exemple `Solde insuffisant : 2000 € demandés, pour un solde de 1000 €`.

Vérifiez qu'un `except MetierError` l'attrape, et affichez son message et son attribut `solde`.

In [ ]:
class MetierError(Exception):
  """Opération refusée par les règles de la banque."""


class PlafondDepasseError(MetierError):
  """Le solde dépasserait le plafond du compte."""


class AccesRefuseError(MetierError):
  """Le demandeur n'est pas titulaire du compte."""


class CompteEnDoubleError(MetierError):
  """Le client a déjà un compte de ce type."""

In [ ]:
# Votre code ici

### Conseils supplémentaires

`SoldeInsuffisantError` hérite de `MetierError`. Son `__init__` range ses deux paramètres dans `self`, puis appelle `super().__init__(message)` : c'est ce message qu'`Exception` garde, et que `str(e)` renvoie. Un `except` attrape l'exception de la classe qu'il nomme et celles de toutes ses sous-classes, et `e` donne accès à ses attributs.

```python
class SoldeInsuffisantError(MetierError):
  """Le solde deviendrait négatif, ou dépasserait le découvert autorisé."""

  def __init__(self, solde: int, montant: int) -> None:
    self.solde = solde
    self.montant = ...
    super().__init__(f"Solde insuffisant : {...} € demandés, pour un solde de {...} €")


try:
  raise SoldeInsuffisantError(1000, 2000)
except MetierError as e:
  print(f"Refusé : {e} (solde : {e.solde} €)")
```

### Solution

In [ ]:
class SoldeInsuffisantError(MetierError):
  """Le solde deviendrait négatif, ou dépasserait le découvert autorisé."""

  def __init__(self, solde: int, montant: int) -> None:
    self.solde = solde
    self.montant = montant
    super().__init__(
      f"Solde insuffisant : {montant} € demandés, pour un solde de {solde} €"
    )


try:
  raise SoldeInsuffisantError(1000, 2000)
except MetierError as e:
  print(f"Refusé : {type(e).__name__} : {e} (solde : {e.solde} €)")

## La classe abstraite `Compte`

Écrivez une classe abstraite `Compte`, la classe de base de tous les comptes :

- `__init__` reçoit le nom du titulaire, range le n-uplet des titulaires (`("Isabelle",)`) dans l'attribut `titulaires`, et un solde de 0 dans l'attribut `_solde` ;
- la propriété `solde` donne le solde, en lecture seule ;
- la méthode abstraite `_verifie_solde(nouveau_solde)` lève une `MetierError` si le compte ne peut pas avoir ce solde ;
- la méthode `_verifie_montant(montant)` lève une `ValueError` si le montant n'est pas strictement positif : un montant nul ou négatif n'est pas un refus prévu par les règles, mais une erreur de programmation de l'appelant ;
- `deposer(montant)`, `retirer(demandeur, montant)` et `virer(demandeur, cible, montant)` font les opérations des spécifications : elles vérifient le montant, et lèvent une `AccesRefuseError` quand le demandeur n'est pas titulaire. `virer` crédite la cible en appelant sa méthode `deposer`, sans toucher à son `_solde` ;
- `__repr__` affiche par exemple `CompteCourant('Isabelle', solde=800)`.

Vérifiez que créer un `Compte` lève une `TypeError` : les opérations se testeront avec les sous-classes de l'exercice suivant.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une propriété sans setter est en lecture seule : `compte.solde = 0` lève une `AttributeError`, alors que les méthodes de la classe modifient `self._solde`. `_verifie_montant` n'utilise ni l'instance ni la classe : c'est une méthode statique. Les opérations vérifient tout avant de modifier quoi que ce soit : `virer` vérifie le demandeur, le montant et le solde de la source, puis appelle `cible.deposer(montant)`, qui fait les vérifications de la cible et lève avant de modifier son solde ; elle ne débite la source qu'ensuite. Un objet ne modifie pas l'attribut protégé d'un autre : il lui envoie un message, en appelant l'une de ses méthodes. Chaque sous-classe définira sa propre règle dans `_verifie_solde`, que les opérations de `Compte` appellent sans savoir de quel type de compte il s'agit. Une méthode `_verifie_titulaire(demandeur)` évite d'écrire deux fois la vérification du demandeur. Pour `__repr__`, `", ".join(repr(t) for t in self.titulaires)` donne `'Isabelle', 'Jean'`.

```python
from abc import ABC, abstractmethod


class Compte(ABC):
  def __init__(self, titulaire: str) -> None:
    self.titulaires = (titulaire,)
    self._solde = 0

  @property
  def solde(self) -> int:
    return ...

  @abstractmethod
  def _verifie_solde(self, nouveau_solde: int) -> None: ...

  @staticmethod
  def _verifie_montant(montant: int) -> None:
    if ...:
      raise ValueError(f"Le montant doit être strictement positif, pas {montant}")

  def _verifie_titulaire(self, demandeur: str) -> None:
    if ...:
      raise AccesRefuseError(f"{demandeur} n'est pas titulaire de ce compte")

  def deposer(self, montant: int) -> None:
    self._verifie_montant(montant)
    self._verifie_solde(self._solde + montant)
    self._solde += montant

  def retirer(self, demandeur: str, montant: int) -> None: ...

  def virer(self, demandeur: str, cible: "Compte", montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_montant(montant)
    self._verifie_solde(...)
    cible.deposer(...)
    ...

  def __repr__(self) -> str:
    titulaires = ", ".join(repr(t) for t in self.titulaires)
    return f"{type(self).__name__}({titulaires}, solde={...})"


try:
  Compte("Isabelle")
except TypeError as e:
  print(e)
```

### Solution

In [ ]:
from abc import ABC, abstractmethod


class Compte(ABC):
  def __init__(self, titulaire: str) -> None:
    self.titulaires = (titulaire,)
    self._solde = 0

  @property
  def solde(self) -> int:
    return self._solde

  @abstractmethod
  def _verifie_solde(self, nouveau_solde: int) -> None:
    """Lève une MetierError si le compte ne peut pas avoir ce solde."""

  @staticmethod
  def _verifie_montant(montant: int) -> None:
    if montant <= 0:
      raise ValueError(f"Le montant doit être strictement positif, pas {montant}")

  def _verifie_titulaire(self, demandeur: str) -> None:
    if demandeur not in self.titulaires:
      raise AccesRefuseError(f"{demandeur} n'est pas titulaire de ce compte")

  def deposer(self, montant: int) -> None:
    self._verifie_montant(montant)
    self._verifie_solde(self._solde + montant)
    self._solde += montant

  def retirer(self, demandeur: str, montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_montant(montant)
    self._verifie_solde(self._solde - montant)
    self._solde -= montant

  def virer(self, demandeur: str, cible: "Compte", montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_montant(montant)
    self._verifie_solde(self._solde - montant)
    cible.deposer(montant)  # Vérifie la cible, et lève avant de la modifier
    self._solde -= montant

  def __repr__(self) -> str:
    titulaires = ", ".join(repr(t) for t in self.titulaires)
    return f"{type(self).__name__}({titulaires}, solde={self._solde})"


try:
  Compte("Isabelle")
except TypeError as e:
  print(f"Refusé : {e}")

## Les types de comptes

Écrivez les quatre types de comptes des spécifications, sous-classes de `Compte` :

- `CompteCourant`, dont l'attribut de classe `decouvert_autorise` vaut 500 ;
- `CompteJoint`, sous-classe de `CompteCourant`, qu'on crée avec ses deux titulaires : `CompteJoint("Isabelle", "Jean")` ;
- `CompteEpargne`, dont le solde ne peut pas être négatif ;
- `LivretA`, sous-classe de `CompteEpargne`, dont l'attribut de classe `plafond` vaut 22 950.

Chacun a aussi un attribut de classe `libelle` (`"compte courant"`, `"compte joint"`…), qui servira à l'exercice suivant. Un solde négatif au-delà du découvert autorisé lève une `SoldeInsuffisantError`, créée avec le solde actuel du compte et le montant demandé, et un solde au-dessus du plafond une `PlafondDepasseError`. Chaque redéfinition de `_verifie_solde` porte le décorateur `@override`.

Testez-les ensuite avec la cellule de vérification qui suit la solution.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Chaque classe ne définit que ce qui la distingue : sa règle dans `_verifie_solde`, ses attributs de classe, et, pour `CompteJoint`, un `__init__` qui appelle `super().__init__(titulaire)` puis ajoute le cotitulaire aux `titulaires`. `CompteJoint` hérite de la règle de `CompteCourant`. Le `_verifie_solde` de `LivretA` appelle d'abord celui de `CompteEpargne` avec `super()`, puis vérifie le plafond. Dans une méthode, `self.decouvert_autorise` lit l'attribut de classe. `_verifie_solde` reçoit le nouveau solde : le montant demandé est l'écart entre le solde actuel et lui, `self._solde - nouveau_solde`. `@override`, du module `typing` (Python 3.12), fait signaler par un vérificateur de types une méthode qui ne redéfinit rien, par exemple à cause d'une faute de frappe dans son nom.

```python
from typing import override


class CompteCourant(Compte):
  libelle = "compte courant"
  decouvert_autorise = 500

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < ...:
      raise SoldeInsuffisantError(self._solde, ...)


class CompteJoint(CompteCourant):
  libelle = "compte joint"

  def __init__(self, titulaire: str, cotitulaire: str) -> None:
    super().__init__(titulaire)
    self.titulaires = ...


class CompteEpargne(Compte): ...


class LivretA(CompteEpargne):
  libelle = "livret A"
  plafond = 22_950

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    super()._verifie_solde(nouveau_solde)
    ...
```

### Solution

In [ ]:
from typing import override


class CompteCourant(Compte):
  libelle = "compte courant"
  decouvert_autorise = 500

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < -self.decouvert_autorise:
      raise SoldeInsuffisantError(self._solde, self._solde - nouveau_solde)


class CompteJoint(CompteCourant):
  libelle = "compte joint"

  def __init__(self, titulaire: str, cotitulaire: str) -> None:
    super().__init__(titulaire)
    self.titulaires = (titulaire, cotitulaire)


class CompteEpargne(Compte):
  libelle = "compte d'épargne"

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < 0:
      raise SoldeInsuffisantError(self._solde, self._solde - nouveau_solde)


class LivretA(CompteEpargne):
  libelle = "livret A"
  plafond = 22_950

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    super()._verifie_solde(nouveau_solde)
    if nouveau_solde > self.plafond:
      raise PlafondDepasseError(
        f"Plafond de {self.plafond} € dépassé : le solde passerait à {nouveau_solde} €"
      )

### Vérification

La cellule suivante rejoue le scénario de la version procédurale avec vos classes, puis tente des opérations que les règles refusent. La fonction `tente` exécute une opération et affiche si elle est acceptée, ou l'exception métier qui la refuse ; d'une `SoldeInsuffisantError`, elle affiche les attributs `montant` et `solde`. Enfin, un dépôt négatif lève une `ValueError`, que `tente` laisse passer : ce n'est pas un refus prévu par les règles, mais une erreur de programmation.

Résultats attendus : des soldes de 800, 1 000, 1 140 et 7 000 € ; la 4e opération acceptée, les autres refusées (2 000 € demandés pour un solde de 1 000 €, `PlafondDepasseError`, `AccesRefuseError`, puis 200 € demandés pour un solde de -400 € : le découvert dépasserait 500 €) ; à la fin, -400 € sur le compte courant et toujours 7 000 € sur le compte joint, que le virement refusé n'a pas touché ; une `AttributeError` quand on affecte le solde, et une `ValueError` pour le dépôt négatif.

In [ ]:
from collections.abc import Callable


def tente(description: str, operation: Callable[[], object]) -> None:
  try:
    operation()
  except SoldeInsuffisantError as e:
    print(
      f"{description} : refusé, {e.montant} € demandés pour un solde de {e.solde} €"
    )
  except MetierError as e:
    print(f"{description} : refusé ({type(e).__name__} : {e})")
  else:
    print(f"{description} : accepté")


courant = CompteCourant("Isabelle")
epargne = CompteEpargne("Jean")
livret = LivretA("Jean")
joint = CompteJoint("Isabelle", "Jean")
courant.deposer(1300)
epargne.deposer(1000)
livret.deposer(1200)
joint.deposer(6500)
courant.virer("Isabelle", joint, 500)
livret.retirer("Jean", 60)
print(courant, epargne, livret, joint, sep="\n")

tente(
  "Jean retire 2000 € de son compte d'épargne", lambda: epargne.retirer("Jean", 2000)
)
tente("Jean dépose 22 000 € sur son livret A", lambda: livret.deposer(22_000))
tente("Jean retire 100 € du compte d'Isabelle", lambda: courant.retirer("Jean", 100))
tente("Isabelle retire 1200 € de son compte", lambda: courant.retirer("Isabelle", 1200))
tente(
  "Isabelle vire 200 € au compte joint", lambda: courant.virer("Isabelle", joint, 200)
)
print(courant, joint, sep="\n")

try:
  courant.solde = 1_000_000
except AttributeError as e:
  print(f"Refusé : {e}")

try:
  courant.deposer(-100)
except ValueError as e:
  print(f"Erreur de programmation : {e}")

## Les clients

Écrivez une classe `Client`, créée avec le nom du client, qui associe un client à ses comptes :

- l'attribut `comptes` est un dictionnaire, vide au départ, dont les clés sont les types de comptes (les classes elles-mêmes, comme `CompteCourant`) et les valeurs les comptes du client ;
- `ajouter(compte)` ajoute un compte au client, en levant une `AccesRefuseError` si le client n'en est pas titulaire, et une `CompteEnDoubleError` s'il a déjà un compte de ce type ;
- la propriété `patrimoine` donne la somme des soldes de ses comptes ;
- `__repr__` affiche le nom du client et la liste de ses comptes.

Un compte existe sans son client, et un compte joint appartient à deux clients : c'est une agrégation, pas une composition. Testez votre classe avec la cellule de vérification qui suit la solution.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`type(compte)` renvoie la classe d'un compte, qui sert de clé : `CompteJoint` et `CompteCourant` sont deux clés différentes, même si l'une hérite de l'autre. `in` teste si une clé est dans un dictionnaire, et `sum` additionne les soldes que produit une expression génératrice sur `self.comptes.values()`. Le message de `CompteEnDoubleError` peut utiliser l'attribut de classe `libelle` du compte : `Isabelle a déjà un compte courant`.

```python
class Client:
  def __init__(self, nom: str) -> None:
    self.nom = nom
    self.comptes: dict[type[Compte], Compte] = {}

  def ajouter(self, compte: Compte) -> None:
    if self.nom not in compte.titulaires:
      raise AccesRefuseError(...)
    if type(compte) in self.comptes:
      raise CompteEnDoubleError(f"{self.nom} a déjà un {compte.libelle}")
    self.comptes[...] = compte

  @property
  def patrimoine(self) -> int:
    return sum(...)

  def __repr__(self) -> str:
    return f"Client({self.nom!r}, comptes={list(...)})"
```

### Solution

In [ ]:
class Client:
  def __init__(self, nom: str) -> None:
    self.nom = nom
    self.comptes: dict[type[Compte], Compte] = {}

  def ajouter(self, compte: Compte) -> None:
    if self.nom not in compte.titulaires:
      raise AccesRefuseError(f"{self.nom} n'est pas titulaire de {compte!r}")
    if type(compte) in self.comptes:
      raise CompteEnDoubleError(f"{self.nom} a déjà un {compte.libelle}")
    self.comptes[type(compte)] = compte

  @property
  def patrimoine(self) -> int:
    return sum(compte.solde for compte in self.comptes.values())

  def __repr__(self) -> str:
    return f"Client({self.nom!r}, comptes={list(self.comptes.values())})"

### Vérification

Résultats attendus : Isabelle a le compte courant et le compte joint, Jean le compte d'épargne, le livret A et le compte joint ; leurs patrimoines sont de 6 600 et 9 140 € (le compte joint compte pour les deux) ; le second compte courant d'Isabelle est refusé (`CompteEnDoubleError`), comme le livret A de Jean (`AccesRefuseError`), mais pas un livret A à son nom.

In [ ]:
isabelle = Client("Isabelle")
jean = Client("Jean")
isabelle.ajouter(courant)
isabelle.ajouter(joint)
jean.ajouter(epargne)
jean.ajouter(livret)
jean.ajouter(joint)
print(isabelle, jean, sep="\n")
print(isabelle.patrimoine, jean.patrimoine)

tente(
  "Isabelle ouvre un second compte courant",
  lambda: isabelle.ajouter(CompteCourant("Isabelle")),
)
tente("Isabelle ajoute le livret A de Jean", lambda: isabelle.ajouter(livret))
tente("Isabelle ouvre un livret A", lambda: isabelle.ajouter(LivretA("Isabelle")))
print(isabelle.comptes[LivretA])

## La banque

Une banque ouvre les comptes de ses clients et les garde : elle les crée elle-même, c'est une composition. Écrivez une classe `Banque` :

- son attribut de classe `TYPES` associe à chaque nom de type de compte (`"courant"`, `"joint"`, `"épargne"` et `"livret A"`) sa classe : c'est la fabrique du cours, un dictionnaire de classes ;
- `ouvrir(type_compte, *clients)` crée un compte du type nommé, au nom du client donné (des deux, pour un compte joint), l'ajoute aux comptes de chaque client et à ceux de la banque, et le renvoie. Si l'un des clients a déjà un compte de ce type, elle lève une `CompteEnDoubleError` sans rien créer ; pour un type inconnu, une `ValueError` ;
- `len(banque)` donne le nombre de comptes ouverts, et `for compte in banque` les parcourt : ce sont les méthodes spéciales `__len__` et `__iter__`.

Testez votre classe avec la cellule de vérification qui suit la solution.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`TYPES` s'écrit dans le corps de la classe, comme `EXPORTEURS` dans la fabrique du cours. `self.TYPES[type_compte]` donne une classe, qu'on appelle comme toute classe pour créer le compte : `classe("Claire")`. Avec `*clients`, `ouvrir` reçoit un ou deux clients, dont les noms sont les titulaires du compte : `classe(*(client.nom for client in clients))` les passe tous. `Client.ajouter` vérifie déjà la règle d'un compte par type, mais vérifiez tous les clients avant de créer quoi que ce soit : si le second titulaire d'un compte joint en a déjà un, le premier ne doit pas en recevoir un nouveau. `__len__` renvoie un entier, et `__iter__` un itérateur, que donne `iter(self._comptes)`.

```python
from collections.abc import Iterator


class Banque:
  TYPES: dict[str, type[Compte]] = {
    "courant": CompteCourant,
    "joint": CompteJoint,
    "épargne": ...,
    "livret A": ...,
  }

  def __init__(self) -> None:
    self._comptes: list[Compte] = []

  def ouvrir(self, type_compte: str, *clients: Client) -> Compte:
    if type_compte not in self.TYPES:
      raise ValueError(f"Type de compte inconnu : {type_compte!r}")
    classe = self.TYPES[type_compte]
    for client in clients:
      if ...:
        raise CompteEnDoubleError(f"{client.nom} a déjà un {classe.libelle}")
    compte = classe(*(client.nom for client in clients))
    ...
    return compte

  def __len__(self) -> int:
    return ...

  def __iter__(self) -> Iterator[Compte]:
    return ...
```

### Solution

In [ ]:
from collections.abc import Iterator


class Banque:
  TYPES: dict[str, type[Compte]] = {
    "courant": CompteCourant,
    "joint": CompteJoint,
    "épargne": CompteEpargne,
    "livret A": LivretA,
  }

  def __init__(self) -> None:
    self._comptes: list[Compte] = []

  def ouvrir(self, type_compte: str, *clients: Client) -> Compte:
    if type_compte not in self.TYPES:
      raise ValueError(f"Type de compte inconnu : {type_compte!r}")
    classe = self.TYPES[type_compte]
    for client in clients:  # Tout vérifier avant de créer quoi que ce soit
      if classe in client.comptes:
        raise CompteEnDoubleError(f"{client.nom} a déjà un {classe.libelle}")
    compte = classe(*(client.nom for client in clients))
    for client in clients:
      client.ajouter(compte)
    self._comptes.append(compte)
    return compte

  def __len__(self) -> int:
    return len(self._comptes)

  def __iter__(self) -> Iterator[Compte]:
    return iter(self._comptes)

### Vérification

La banque ouvre les comptes de trois nouveaux clients, Claire, Marc et Léa, puis on tente d'en ouvrir d'autres.

Résultats attendus : le second compte courant de Claire refusé (`CompteEnDoubleError`), comme le compte joint de Léa et Marc, qui en a déjà un ; le type « Livret A », mal orthographié, refusé par une `ValueError` ; la banque compte 4 comptes, dont l'épargne de Marc, crédité de 500 € ; et Léa n'a aucun compte, bien que le refus soit venu de Marc.

In [ ]:
banque = Banque()
claire = Client("Claire")
marc = Client("Marc")
lea = Client("Léa")
banque.ouvrir("courant", claire)
banque.ouvrir("livret A", claire)
banque.ouvrir("joint", claire, marc)
banque.ouvrir("épargne", marc).deposer(500)

tente("Claire ouvre un second compte courant", lambda: banque.ouvrir("courant", claire))
tente("Léa ouvre un compte joint avec Marc", lambda: banque.ouvrir("joint", lea, marc))
try:
  banque.ouvrir("Livret A", lea)
except ValueError as e:
  print(f"Erreur de programmation : {e}")

print(len(banque))
for compte in banque:
  print(compte)
print(claire, marc, lea, sep="\n")

## L'historique des opérations (facultatif)

Cette dernière étape est facultative : sautez-la si le temps vous manque. Chaque compte garde l'historique des opérations qu'il a acceptées :

- une dataclass `Operation`, figée (`frozen=True`), décrit une opération par sa nature (`"dépôt"`, `"retrait"` ou `"virement"`), son montant et le solde du compte après l'opération ;
- l'attribut `historique` d'un compte est une liste d'`Operation`, vide à sa création, à laquelle `deposer`, `retirer` et `virer` ajoutent l'opération qu'elles viennent de faire. Un virement s'inscrit comme un virement sur le compte débité, et comme un dépôt sur le compte crédité ;
- une opération refusée n'apparaît dans aucun historique.

Il faut pour cela modifier `Compte`, puis redéfinir ses quatre sous-classes, même sans les changer : une classe hérite de la classe mère qui existait quand on l'a définie, pas de celle qui porte ce nom aujourd'hui. Vérifiez enfin qu'on ne peut pas modifier une `Operation`.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`@dataclass(frozen=True)`, du module `dataclasses`, écrit `__init__`, `__repr__` et `__eq__` à partir des champs annotés, et interdit de les modifier après la création : une affectation lève une `FrozenInstanceError`, sous-classe d'`AttributeError`. Chaque opération crée une `Operation` après avoir modifié le solde, et l'ajoute à l'historique avec `append`. Dans `virer`, `cible.deposer(montant)` inscrit déjà le dépôt dans l'historique de la cible. Attendu de la vérification : sur le compte courant, un dépôt de 1 000 €, un virement de 300 € et un retrait de 50 €, pour un solde final de 650 € ; sur le livret A, un dépôt de 300 €.

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Operation:
  nature: str
  ...


class Compte(ABC):
  def __init__(self, titulaire: str) -> None:
    self.titulaires = (titulaire,)
    self._solde = 0
    self.historique: list[Operation] = []

  ...  # solde, _verifie_solde, _verifie_montant et _verifie_titulaire sans changement

  def deposer(self, montant: int) -> None:
    self._verifie_montant(montant)
    self._verifie_solde(self._solde + montant)
    self._solde += montant
    self.historique.append(Operation("dépôt", montant, self._solde))

  ...  # retirer et virer de même, puis __repr__


...  # CompteCourant, CompteJoint, CompteEpargne et LivretA, recopiés tels quels
```

### Solution

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import override


@dataclass(frozen=True)
class Operation:
  nature: str
  montant: int
  solde: int  # Le solde du compte après l'opération


class Compte(ABC):
  def __init__(self, titulaire: str) -> None:
    self.titulaires = (titulaire,)
    self._solde = 0
    self.historique: list[Operation] = []

  @property
  def solde(self) -> int:
    return self._solde

  @abstractmethod
  def _verifie_solde(self, nouveau_solde: int) -> None:
    """Lève une MetierError si le compte ne peut pas avoir ce solde."""

  @staticmethod
  def _verifie_montant(montant: int) -> None:
    if montant <= 0:
      raise ValueError(f"Le montant doit être strictement positif, pas {montant}")

  def _verifie_titulaire(self, demandeur: str) -> None:
    if demandeur not in self.titulaires:
      raise AccesRefuseError(f"{demandeur} n'est pas titulaire de ce compte")

  def deposer(self, montant: int) -> None:
    self._verifie_montant(montant)
    self._verifie_solde(self._solde + montant)
    self._solde += montant
    self.historique.append(Operation("dépôt", montant, self._solde))

  def retirer(self, demandeur: str, montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_montant(montant)
    self._verifie_solde(self._solde - montant)
    self._solde -= montant
    self.historique.append(Operation("retrait", montant, self._solde))

  def virer(self, demandeur: str, cible: "Compte", montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_montant(montant)
    self._verifie_solde(self._solde - montant)
    cible.deposer(montant)  # Inscrit le dépôt dans l'historique de la cible
    self._solde -= montant
    self.historique.append(Operation("virement", montant, self._solde))

  def __repr__(self) -> str:
    titulaires = ", ".join(repr(t) for t in self.titulaires)
    return f"{type(self).__name__}({titulaires}, solde={self._solde})"


# Les sous-classes, recopiées telles quelles, pour qu'elles héritent du nouveau Compte
class CompteCourant(Compte):
  libelle = "compte courant"
  decouvert_autorise = 500

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < -self.decouvert_autorise:
      raise SoldeInsuffisantError(self._solde, self._solde - nouveau_solde)


class CompteJoint(CompteCourant):
  libelle = "compte joint"

  def __init__(self, titulaire: str, cotitulaire: str) -> None:
    super().__init__(titulaire)
    self.titulaires = (titulaire, cotitulaire)


class CompteEpargne(Compte):
  libelle = "compte d'épargne"

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < 0:
      raise SoldeInsuffisantError(self._solde, self._solde - nouveau_solde)


class LivretA(CompteEpargne):
  libelle = "livret A"
  plafond = 22_950

  @override
  def _verifie_solde(self, nouveau_solde: int) -> None:
    super()._verifie_solde(nouveau_solde)
    if nouveau_solde > self.plafond:
      raise PlafondDepasseError(
        f"Plafond de {self.plafond} € dépassé : le solde passerait à {nouveau_solde} €"
      )

### Vérification

Résultats attendus : le virement de 2 000 € refusé (solde insuffisant) ; trois opérations sur le compte courant, de `Operation(nature='dépôt', montant=1000, solde=1000)` à un retrait de 50 € qui laisse un solde de 650 € ; un seul dépôt de 300 € sur le livret A ; et une `FrozenInstanceError` quand on modifie une opération.

In [ ]:
courant = CompteCourant("Isabelle")
livret = LivretA("Isabelle")
courant.deposer(1000)
courant.virer("Isabelle", livret, 300)
courant.retirer("Isabelle", 50)
tente(
  "Isabelle vire 2000 € sur son livret A",
  lambda: courant.virer("Isabelle", livret, 2000),
)
for operation in courant.historique:
  print(operation)
print(livret.historique)

try:
  courant.historique[0].montant = 1_000_000
except AttributeError as e:
  print(f"Refusé : {type(e).__name__} : {e}")